# prototype clock value in jax
Make a working clock-value code with jax and jax-finufft that is fast enough!

## author:
- **David W. Hogg** (NYU) (Flatiron) (MPIA)

## dependencies:
- *KeplerClocks* at https://github.com/davidwhogg/KeplerClocks

## bugs:
- There should be a plot of the data, a plot of the L-S output, and a plot of the clock values before filtering.
- Maybe, when a light curve contains multiple clocks, the clocks should be fit simultaneously and the empirical clock value computed in that context?
- Requires that *KeplerClocks* be checked out at the same level as *ClockValue*. That's dumb.

## comments:
- Because my Mac has issues with jax, only use jax when we absolutely need it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
from pathlib import Path
import sys
target_dir = Path.cwd() / "../../KeplerClocks/py"
sys.path.append(str(target_dir.resolve()))
import clocks

In [ ]:
# set high-level parameters of the method
MAX_PERIOD = 30. # days
MIN_VALUE = 1.e7 # inverse days squared
MIN_OPTIMISTIC_VALUE = 1.e9 # inverse days squared

In [ ]:
# clock plotting functions

def latex_sci_not(x):
    if x == 0:
        return "$0$"
    s = f"{x:.1e}"
    mantissa, exponent = s.split("e")
    return rf"${mantissa} \times 10^{{{int(exponent)}}}$"# look at best frequency

def set_plot_range(xs):
    a, b = np.percentile(xs, [0.025, 97.5])
    mid, dif = 0.5 * (b + a), 0.5 * (b - a)
    return mid - 2. * dif, mid + 2 * dif

def plot_clock(kicid, om, M, val, optval, theoval, ts, ys, ivs):
    """
    ## bugs
    - Note commented-out code that plots model derivative.
    """
    period = 2. * np.pi / om
    f = plt.figure(figsize=(9, 3))
    _, ms, pars = clocks.fourier_wls_fit(om, ts, ys, ivs, M)
    # dpars = clocks.take_derivative_wrt_phase(pars, ms)
    thetas_plot = np.linspace(0., 4. * np.pi, 1000)
    X_plot, _ = clocks.design_matrix(om, thetas_plot / om, M)
    plt.scatter((om * ts) % (2. * np.pi), ys, s=1, c="k", marker=".", alpha=0.5)
    plt.scatter((om * ts) % (2. * np.pi) + 2. * np.pi, ys, s=1, c="k", marker=".")
    plt.plot(thetas_plot, X_plot @ pars, "r-")
    # plt.plot(thetas_plot, X_plot @ dpars + pars[0], "r--")
    plt.title(f"{kicid}; period {period:.6f} d; value " + latex_sci_not(val)
              + r" d$^{-2}$ (emp) " + latex_sci_not(optval) + " (opt) " + latex_sci_not(theoval) + " (theo)")
    plt.xlim(0., 4. * np.pi)
    plt.ylim(set_plot_range(ys))
    plt.xlabel("phase [rad]")
    return f

In [ ]:
# put it all together into one huge function.

def best_clocks_in_star(kicid, Mmax=128, plot=True):
    print(f"best_clocks_in_star(): getting Kepler data for {kicid}")
    foo = clocks.get_kepler_data(kicid)
    if foo is None:
        print(f"best_clocks_in_star(): skipping {kicid}")
        return [], [], [], []
    ts, ys, errs, deltaf, deltat = foo
    ivars = 1. / errs ** 2

    print(f"best_clocks_in_star(): getting candidate frequencies for {kicid}")
    candidate_oms = 2. * np.pi * clocks.get_candidate_frequencies(ts, ys, errs, deltaf, deltat)
    
    # now loop through candidates and refine them
    oms, values = np.zeros_like(candidate_oms), np.zeros_like(candidate_oms)
    Ms = np.zeros_like(candidate_oms).astype(int)
    print(f"best_clocks_in_star(): getting clock values for {kicid}")
    for i, om0 in enumerate(candidate_oms):
        oms[i], values[i], Ms[i] = clocks.get_best_clock(om0, ts, ys, ivars, Mmax, deltaf, deltat)
    optimistic_values = np.array([clocks.optimistic_clock_value(om, ts, ys, ivars, M) for om, M in zip(oms, Ms)])
    theoretical_values = np.array([clocks.theoretical_clock_value(om, ts, ys, ivars, M) for om, M in zip(oms, Ms)])
    
    # now filter and arrange the clocks
    good = (values > MIN_VALUE) | (optimistic_values > MIN_OPTIMISTIC_VALUE)
    if np.sum(good) < 1:
        return [], [], [], []
    oms, values, Ms, optimistic_values, theoretical_values = \
        oms[good], values[good], Ms[good], optimistic_values[good], theoretical_values[good]
    idx_sort = np.argsort(values)[::-1]
    oms, values, Ms, optimistic_values, theoretical_values = \
        oms[idx_sort], values[idx_sort], Ms[idx_sort], optimistic_values[idx_sort], theoretical_values[idx_sort]
    idx_remove = np.logical_not(clocks.identify_resonances(oms, np.pi * deltaf)) # slightly made up?
    oms, values, Ms, optimistic_values, theoretical_values = \
        oms[idx_remove], values[idx_remove], Ms[idx_remove], optimistic_values[idx_remove], theoretical_values[idx_remove]

    if plot:
        for om, value, optval, theoval, M in zip(oms, values, optimistic_values, theoretical_values, Ms):
            f = plot_clock(kicid, om, M, value, optval, theoval, ts, ys, ivars)
            plt.show()
    return oms, values, optimistic_values, Ms

In [ ]:
# now do a whole bunch of KICs

kics = [ "KIC002162994", "KIC005285607", "KIC003240411",
         "KIC003459297", "KIC003865742", "KIC004930889",
    "KIC004939281", "KIC005309849", "KIC005941844", "KIC006352430", "KIC011013201",
    "KIC006462033", "KIC006780397", "KIC007630417", "KIC007760680", "KIC008057661",
    "KIC008255796", "KIC008381949", "KIC008459899", "KIC008714886", "KIC008766405",
    "KIC009020774", "KIC009715425", "KIC010526294", "KIC011360704", "KIC011971405",
    "KIC012258330", "KIC000520290", "KIC009111849", "KIC009845898", "KIC006116172",
    "KIC005617259", "KIC009289704", "KIC007767699", "KIC008249829", "KIC010874614",
    "KIC009895543" ]
kics_interesting = [ "KIC005128966", ]
kics_fail =  [ "KIC005112738", "KIC005112705", "KIC005112855", "KIC012251995" ]
kics_boring = [ "KIC009041983", "KIC007022977", "KIC009224229" ]
kics_dumb =  [ "KIC007264961", "KIC009656543", "KIC009772694", "KIC010416004",
               "KIC004936089", "KIC010581918", "KIC010992414", ]
kics_pulse = [ "KIC010056297", "KIC006587551", "KIC003544595", "KIC011295426",
               "KIC009700322", "KIC006370665", "KIC002581626", "KIC009828226",
               "KIC008309815", "KIC008290073", "KIC007840896" ]
kics_seismic = [ "KIC010407873", "KIC011551430", ]
for kicid in [ "KIC011234677", ]:
    oms, vals, optvals, Ms = best_clocks_in_star(kicid)